In [2]:
# Import required libraries
import h2o
from h2o.automl import H2OAutoML
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix
import warnings

warnings.filterwarnings("ignore")

# Initialize H2O
print("Initializing H2O...")
h2o.init(max_mem_size="4G")  # Adjust based on available memory
print("H2O initialized successfully!")

Initializing H2O...
Checking whether there is an H2O instance running at http://localhost:54321. connected.
Please download and install the latest version from: https://h2o-release.s3.amazonaws.com/h2o/latest_stable.html


H2O_cluster_uptime:,1 day 3 hours 41 mins
H2O_cluster_timezone:,Europe/Oslo
H2O_data_parsing_timezone:,UTC
H2O_cluster_version:,3.46.0.7
H2O_cluster_version_age:,5 months and 20 days
H2O_cluster_name:,H2O_from_python_kristsey_jmb5ae
H2O_cluster_total_nodes:,1
H2O_cluster_free_memory:,1.078 Gb
H2O_cluster_total_cores:,24
H2O_cluster_allowed_cores:,24
H2O_cluster_status:,"locked, healthy"


H2O initialized successfully!


In [3]:
# Load the historical flights dataset
print("Loading historical flights dataset...")
data_path = '/home/kristsey/AvinorHackathon/data/historical_flights.csv'

# Load data in chunks due to large size
chunk_size = 50000
chunks = []
total_rows = 0

for chunk in pd.read_csv(data_path, chunksize=chunk_size):
    chunks.append(chunk)
    total_rows += len(chunk)
    print(f"Loaded {total_rows:,} rows...")

# Combine chunks
df = pd.concat(chunks, ignore_index=True)
print(f"Final dataset shape: {df.shape}")

# Display basic information about the dataset
print("\nDataset Info:")
print(df.info())
print("\nFirst few rows:")
print(df.head())
print("\nDataset statistics:")
print(df.describe())

Loading historical flights dataset...
Loaded 50,000 rows...
Loaded 100,000 rows...
Loaded 150,000 rows...
Loaded 200,000 rows...
Loaded 250,000 rows...
Loaded 300,000 rows...
Loaded 350,000 rows...
Loaded 400,000 rows...
Loaded 410,442 rows...
Final dataset shape: (410442, 11)

Dataset Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 410442 entries, 0 to 410441
Data columns (total 11 columns):
 #   Column             Non-Null Count   Dtype 
---  ------             --------------   ----- 
 0   flight_id          410442 non-null  object
 1   dep_airport        410442 non-null  object
 2   dep_airport_group  232968 non-null  object
 3   arr_airport        410442 non-null  object
 4   arr_airport_group  231669 non-null  object
 5   service_type       410336 non-null  object
 6   std                410442 non-null  object
 7   sta                410442 non-null  object
 8   cancelled          410442 non-null  int64 
 9   atd                394327 non-null  object
 10  ata            

In [7]:
# Data Preprocessing and Feature Engineering
print("Starting data preprocessing...")

# Create a copy for processing
df_processed = df.copy()

# Handle missing values
print(f"Missing values per column:\n{df_processed.isnull().sum()}")

# Convert datetime columns
datetime_cols = ["std", "sta", "atd", "ata"]
for col in datetime_cols:
    if col in df_processed.columns:
        df_processed[col] = pd.to_datetime(df_processed[col], errors="coerce")


# Create target variable: flight delay (binary classification)
# A flight is considered delayed if actual departure time (atd) is more than 15 minutes after scheduled departure time (std)
def calculate_delay(row):
    if pd.isna(row["atd"]) or pd.isna(row["std"]):
        return None
    delay_minutes = (row["atd"] - row["std"]).total_seconds() / 60
    return delay_minutes


df_processed["delay_minutes"] = df_processed.apply(calculate_delay, axis=1)

# Remove rows where we can't calculate delay
df_processed = df_processed.dropna(subset=["delay_minutes"])
print(f"Dataset shape after removing rows without delay info: {df_processed.shape}")

# Feature engineering
# Extract time features from scheduled departure time
df_processed["dep_hour"] = df_processed["std"].dt.hour
df_processed["dep_day_of_week"] = df_processed["std"].dt.dayofweek
df_processed["dep_month"] = df_processed["std"].dt.month
df_processed["dep_day_of_year"] = df_processed["std"].dt.dayofyear

# Extract time features from scheduled arrival time
df_processed["arr_hour"] = df_processed["sta"].dt.hour

# Calculate scheduled flight duration in minutes
df_processed["scheduled_duration"] = (
    df_processed["sta"] - df_processed["std"]
).dt.total_seconds() / 60

# Circular encoding for time-based features
print("Creating circular encodings for time-based features...")

# Hour of day (0-23) - circular encoding
df_processed["dep_hour_sin"] = np.sin(2 * np.pi * df_processed["dep_hour"] / 24)
df_processed["dep_hour_cos"] = np.cos(2 * np.pi * df_processed["dep_hour"] / 24)
df_processed["arr_hour_sin"] = np.sin(2 * np.pi * df_processed["arr_hour"] / 24)
df_processed["arr_hour_cos"] = np.cos(2 * np.pi * df_processed["arr_hour"] / 24)

# Day of week (0-6) - circular encoding
df_processed["dep_day_of_week_sin"] = np.sin(
    2 * np.pi * df_processed["dep_day_of_week"] / 7
)
df_processed["dep_day_of_week_cos"] = np.cos(
    2 * np.pi * df_processed["dep_day_of_week"] / 7
)

# Month (1-12) - circular encoding
# Note: months are 1-indexed, so we subtract 1 before encoding
df_processed["dep_month_sin"] = np.sin(2 * np.pi * (df_processed["dep_month"] - 1) / 12)
df_processed["dep_month_cos"] = np.cos(2 * np.pi * (df_processed["dep_month"] - 1) / 12)

# Day of year (1-366) - circular encoding
# Note: day of year is 1-indexed, so we subtract 1 before encoding
df_processed["dep_day_of_year_sin"] = np.sin(
    2 * np.pi * (df_processed["dep_day_of_year"] - 1) / 366
)
df_processed["dep_day_of_year_cos"] = np.cos(
    2 * np.pi * (df_processed["dep_day_of_year"] - 1) / 366
)

print("Circular encodings created successfully!")
print(
    "Added features: dep/arr_hour_sin/cos, dep/arr_day_of_week_sin/cos, dep/arr_month_sin/cos, dep/arr_day_of_year_sin/cos"
)

print("Removing non-circular original time features...")
df_processed = df_processed.drop(
    columns=[
        "dep_hour",
        "dep_day_of_week",
        "dep_month",
        "dep_day_of_year",
        "arr_hour",
        "arr_day_of_week",
        "arr_month",
        "arr_day_of_year",
    ],
    errors="ignore",
)

# Display feature engineering results
print(f"Target variable distribution:")
print(f"Not delayed (0): {(df_processed['delay_minutes'] < 15).sum()}")
print(f"Delayed (1): {(df_processed['delay_minutes'] >=15).sum()}")
print(f"Delay rate: {(df_processed['delay_minutes'] >=15).mean():.2%}")

print(f"\nProcessed dataset shape: {df_processed.shape}")

# Show sample of processed data with circular features
print("\nSample of processed data with circular encodings:")
print(
    df_processed.head(10)
)
print("Processed data columns:")
print(df_processed.columns.tolist())

Starting data preprocessing...
Missing values per column:
flight_id                 0
dep_airport               0
dep_airport_group    177474
arr_airport               0
arr_airport_group    178773
service_type            106
std                       0
sta                       0
cancelled                 0
atd                   16115
ata                   16866
dtype: int64
Dataset shape after removing rows without delay info: (394327, 12)
Creating circular encodings for time-based features...
Circular encodings created successfully!
Added features: dep/arr_hour_sin/cos, dep/arr_day_of_week_sin/cos, dep/arr_month_sin/cos, dep/arr_day_of_year_sin/cos
Removing non-circular original time features...
Target variable distribution:
Not delayed (0): 342842
Delayed (1): 51485
Delay rate: 13.06%

Processed dataset shape: (394327, 23)

Sample of processed data with circular encodings:
   flight_id dep_airport dep_airport_group arr_airport arr_airport_group  \
8     SK4517         MOL          